# ambient-core — notebook run-me (Bronze → Silver → Gold)

Local or generic Jupyter path. **No Databricks.** Clone this repository,
set `PLATFORM_ROOT` to the repo root, copy external `data/raw/` samples,
and run pandas Bronze → Silver → Gold via `ambient_pipeline`.

Copy demo CSVs from EngineerID/ambient-systems `demos/ambient-core/`
into `data/raw/` (not vendored in this repository).

Commercial / UC / Autoloader notebooks are optional — see `notebooks/COMMERCIAL_ONLY.txt`.


In [ ]:
# Repo root — defaults to cwd when this notebook lives under notebooks/
from pathlib import Path
PLATFORM_ROOT = str(Path.cwd().parent if (Path.cwd() / "00_colab_run_me.ipynb").exists() else Path.cwd())
ORG_ID = "demo-org-allmanufacturingds"

%pip install -q pandas PyYAML


In [ ]:
import sys
from pathlib import Path

root = Path(PLATFORM_ROOT).resolve()
if not (root / "ambient_pipeline").is_dir():
  for base in (Path.cwd(), Path.cwd().parent):
    if (base / "ambient_pipeline").is_dir():
      root = base.resolve()
      break
  else:
    raise FileNotFoundError(
      "ambient-core root not found. Set PLATFORM_ROOT to your clone root."
    )

sys.path.insert(0, str(root))
lib = root / "lib"
if lib.is_dir():
  sys.path.insert(0, str(lib))
print("PLATFORM_ROOT =", root)
print("children:", sorted(p.name for p in root.iterdir()))


In [ ]:
from ambient_pipeline.colab_bootstrap import ensure_package_on_path
from ambient_pipeline.colab_smoke import run_colab_smoke, DEMO_TABLES

root = ensure_package_on_path(root)
raw = root / "data" / "raw"
print("Demo tables expected:")
for tid, fn in DEMO_TABLES.items():
    p = raw / fn
    print(f"  {tid:20} {'OK' if p.is_file() else 'MISSING':7} {fn}")

In [ ]:
result = run_colab_smoke(root, org_id=ORG_ID)
print("run_id:", result.run_id)
print("bronze:", result.bronze_tables)
print("silver:", result.silver_tables)
print("gold KPIs:")
for k, v in result.gold_kpis.items():
    print(f"  {k}: {v}")
print("notes:")
for n in result.notes:
    print(" -", n)
print("wrote →", result.output_dir)

In [ ]:
import pandas as pd

gold = root / "data" / "processed" / "gold" / "demo_kpis.csv"
if gold.is_file():
    display(pd.read_csv(gold))
else:
    print("No gold KPIs yet — place Allmanufacturingds-*.csv under data/raw/ and re-run.")

bronze_sample = root / "data" / "processed" / "bronze" / "ar_aging.csv"
if bronze_sample.is_file():
    print("\nBronze AR (provenance columns present):")
    display(pd.read_csv(bronze_sample).head())

## Next

- Place `Allmanufacturingds-*.csv` samples from Drive (or Ambient Systems demo assets) into `data/raw/`.
- Optional Spark / Delta local path: `lib/ambient_pipeline/` + `pip install -e ".[pipeline]"` (needs external CSVs).
- Skip commercial-only notebooks listed in `notebooks/COMMERCIAL_ONLY.txt`.
